Install required gradient boosting libraries: LightGBM, XGBoost, and CatBoost.

In [2]:
!pip install lightgbm xgboost catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


Import core libraries for data processing, encoding, model evaluation, and the three boosting models. Set file paths for train, test, and submission CSVs.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

train_path = './train.csv'
test_path  = './test.csv'
sub_path   = './submission.csv'

Load train and test CSVs. Train has 77,299 rows and 11 columns including the target `demand`. Test has 41,778 rows and 10 columns.

In [6]:
train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)
print(train.shape, test.shape)

(77299, 11) (41778, 10)


Decode each geohash string into (latitude, longitude) coordinates using the base-32 bit-interleaving algorithm. Results are cached in a dictionary for fast lookup.

In [16]:
def decode_geohash(geohash):
    base32   = '0123456789bcdefghjkmnpqrstuvwxyz'
    char_map = {c: i for i, c in enumerate(base32)}
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    for char in geohash:
        val = char_map[char]
        for mask in [16, 8, 4, 2, 1]:
            bit = 1 if (val & mask) else 0
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2
                lon_interval = (mid, lon_interval[1]) if bit else (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2
                lat_interval = (mid, lat_interval[1]) if bit else (lat_interval[0], mid)
            is_even = not is_even
    return (lat_interval[0] + lat_interval[1]) / 2, (lon_interval[0] + lon_interval[1]) / 2

unique_ghs = pd.concat([train['geohash'], test['geohash']]).unique()
gh_coords  = {gh: decode_geohash(gh) for gh in unique_ghs}

Fill missing values in road properties (RoadType, NumberofLanes, LargeVehicles, Landmarks) using the most common value per geohash. Falls back to the global mode if a geohash has no valid observations.

In [8]:
df_all      = pd.concat([train.drop(columns=['demand']), test], axis=0, ignore_index=True)
static_cols = ['RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks']

gh_modes, global_modes = {}, {}
for col in static_cols:
    global_modes[col] = df_all[col].mode()[0]
    gh_modes[col] = df_all.groupby('geohash')[col].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan).to_dict()

def impute_static(df):
    for col in static_cols:
        df[col] = df[col].fillna(df['geohash'].map(gh_modes[col])).fillna(global_modes[col])
    return df

train = impute_static(train)
test  = impute_static(test)

Parse the HH:MM timestamp into minutes since midnight, then apply sine/cosine encoding to capture the circular nature of time. Attach decoded lat/lon and geohash prefix characters as spatial features.

In [9]:
def process_time(df):
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['minutes']  = parts[0] * 60 + parts[1]
    df['sin_time'] = np.sin(2 * np.pi * df['minutes'] / 1440.0)
    df['cos_time'] = np.cos(2 * np.pi * df['minutes'] / 1440.0)
    return df

train = process_time(train)
test  = process_time(test)

for df in [train, test]:
    df['lat']           = df['geohash'].map(lambda x: gh_coords[x][0])
    df['lon']           = df['geohash'].map(lambda x: gh_coords[x][1])
    df['geohash_char4'] = df['geohash'].str[3]
    df['geohash_char5'] = df['geohash'].str[4]

Fill missing Temperature using the mean value at the same (day, time) combination. Fill missing Weather using the most common label at the same timestamp. Both fall back to Day 48 values, then global statistics.

In [10]:
temp_map         = train.groupby(['day', 'minutes'])['Temperature'].mean().to_dict()
global_mean_temp = train['Temperature'].mean()

def impute_temp(row):
    if pd.isna(row['Temperature']):
        k = (row['day'], row['minutes'])
        if k in temp_map: return temp_map[k]
        k_48 = (48, row['minutes'])
        if k_48 in temp_map: return temp_map[k_48]
        return global_mean_temp
    return row['Temperature']

train['Temperature'] = train.apply(impute_temp, axis=1)
test['Temperature']  = test.apply(impute_temp, axis=1)

weather_map = train.groupby(['day', 'minutes'])['Weather'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan).to_dict()
global_mode_weather = train['Weather'].mode()[0]

def impute_weather(row):
    if pd.isna(row['Weather']):
        k = (row['day'], row['minutes'])
        if k in weather_map and not pd.isna(weather_map[k]): return weather_map[k]
        k_48 = (48, row['minutes'])
        if k_48 in weather_map and not pd.isna(weather_map[k_48]): return weather_map[k_48]
        return global_mode_weather
    return row['Weather']

train['Weather'] = train.apply(impute_weather, axis=1)
test['Weather']  = test.apply(impute_weather, axis=1)

Use Day 48 as a reference day to compute per-geohash demand statistics (mean, std, max, min). Build a nearest-neighbour map so geohashes absent from Day 48 borrow stats from their closest geographic neighbour.

In [11]:
day48       = train[train['day'] == 48].copy()
day48_pivot = day48.pivot(index='geohash', columns='minutes', values='demand')
day48_ghs   = day48_pivot.index.tolist()
day48_coords_arr = np.array([gh_coords[gh] for gh in day48_ghs])

def get_nearest_day48_gh(gh):
    if gh in day48_pivot.index: return gh
    lat, lon = gh_coords[gh]
    dists = np.sum((day48_coords_arr - np.array([lat, lon])) ** 2, axis=1)
    return day48_ghs[np.argmin(dists)]

nearest_day48_map = {gh: get_nearest_day48_gh(gh) for gh in unique_ghs}

gh_list       = list(unique_ghs)
gh_coords_arr = np.array([gh_coords[gh] for gh in gh_list])
nearest_neighbors = {}
for gh in gh_list:
    lat, lon    = gh_coords[gh]
    dists       = np.sum((gh_coords_arr - np.array([lat, lon])) ** 2, axis=1)
    nearest_idx = np.argsort(dists)[:4]
    nearest_neighbors[gh] = [gh_list[i] for i in nearest_idx if gh_list[i] != gh][:3]

global_mean_by_min = day48.groupby('minutes')['demand'].mean().to_dict()
global_std_by_min  = day48.groupby('minutes')['demand'].std().to_dict()
gh_day48_mean      = day48.groupby('geohash')['demand'].mean().to_dict()
gh_day48_std       = day48.groupby('geohash')['demand'].std().to_dict()
gh_day48_max       = day48.groupby('geohash')['demand'].max().to_dict()
gh_day48_min       = day48.groupby('geohash')['demand'].min().to_dict()
global_day48_mean  = day48['demand'].mean()

for df in [train, test]:
    df['gh_day48_mean'] = df['geohash'].map(lambda x: gh_day48_mean.get(nearest_day48_map[x], global_day48_mean))
    df['gh_day48_std']  = df['geohash'].map(lambda x: gh_day48_std.get(nearest_day48_map[x], 0.0))
    df['gh_day48_max']  = df['geohash'].map(lambda x: gh_day48_max.get(nearest_day48_map[x], global_day48_mean))
    df['gh_day48_min']  = df['geohash'].map(lambda x: gh_day48_min.get(nearest_day48_map[x], global_day48_mean))

Create 8 lag features per row using Day 48 demand as a 24-hour lookback. Includes exact lag, ±15 and ±30 minute offsets, spatial neighbour average, and global network-wide mean/std. Day 48 rows are set to NaN to prevent data leakage.

In [12]:
def add_lags(df):
    cols = {k: [] for k in ['lag_24h','lag_24h_prev15','lag_24h_next15','lag_24h_prev30','lag_24h_next30','spatial_lag_24h','global_lag_24h_mean','global_lag_24h_std']}
    for _, row in df.iterrows():
        if row['day'] == 48:
            for k in cols: cols[k].append(np.nan)
        else:
            gh, t  = row['geohash'], row['minutes']
            mgh    = nearest_day48_map[gh]
            get    = lambda g, time: day48_pivot.at[g, time] if time in day48_pivot.columns else np.nan
            cols['lag_24h'].append(get(mgh, t))
            cols['lag_24h_prev15'].append(get(mgh, (t-15)%1440))
            cols['lag_24h_next15'].append(get(mgh, (t+15)%1440))
            cols['lag_24h_prev30'].append(get(mgh, (t-30)%1440))
            cols['lag_24h_next30'].append(get(mgh, (t+30)%1440))
            nvals = [get(nearest_day48_map[n], t) for n in nearest_neighbors[gh]]
            nvals = [v for v in nvals if not pd.isna(v)]
            cols['spatial_lag_24h'].append(np.mean(nvals) if nvals else np.nan)
            cols['global_lag_24h_mean'].append(global_mean_by_min.get(t, np.nan))
            cols['global_lag_24h_std'].append(global_std_by_min.get(t, np.nan))
    for k, v in cols.items():
        df[k] = v
    return df

print('Lagging train...')
train = add_lags(train)
print('Lagging test...')
test  = add_lags(test)
print('Done.')

Lagging train...
Lagging test...
Done.


Label-encode all categorical string columns. Encoder is fit on the combined train+test data to handle all possible label values. Final feature set contains 25 columns.

In [13]:
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geohash_char4', 'geohash_char5']
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col].astype(str), test[col].astype(str)]))
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

features = [c for c in train.columns if c not in ['Index','geohash','day','timestamp','demand']]
print(f'{len(features)} features:', features)

25 features: ['RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'minutes', 'sin_time', 'cos_time', 'lat', 'lon', 'geohash_char4', 'geohash_char5', 'gh_day48_mean', 'gh_day48_std', 'gh_day48_max', 'gh_day48_min', 'lag_24h', 'lag_24h_prev15', 'lag_24h_next15', 'lag_24h_prev30', 'lag_24h_next30', 'spatial_lag_24h', 'global_lag_24h_mean', 'global_lag_24h_std']


Time-aware split: train on Day 48 + first hour of Day 49, validate on the rest of Day 49. Train and evaluate LightGBM, XGBoost, and CatBoost independently, then average their predictions into an ensemble. Result: R² = 0.9277.

In [14]:
train_cv = train[(train['day']==48)|((train['day']==49)&(train['minutes']<60))].reset_index(drop=True)
val_cv   = train[(train['day']==49)&(train['minutes']>=60)].reset_index(drop=True)

X_tr, y_tr = train_cv[features], train_cv['demand']
X_vl, y_vl = val_cv[features],   val_cv['demand']

lgb_cv = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=-1, num_leaves=63, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
lgb_cv.fit(X_tr, y_tr); p_lgb = lgb_cv.predict(X_vl)

xgb_cv = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=7, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_cv.fit(X_tr, y_tr); p_xgb = xgb_cv.predict(X_vl)

cat_cv = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=7, random_seed=42, verbose=0)
cat_cv.fit(X_tr, y_tr); p_cat = cat_cv.predict(X_vl)

p_ens = (p_lgb + p_xgb + p_cat) / 3.0
print(f'LGB: {r2_score(y_vl,p_lgb):.5f} | XGB: {r2_score(y_vl,p_xgb):.5f} | CAT: {r2_score(y_vl,p_cat):.5f} | ENS: {r2_score(y_vl,p_ens):.5f}')

LGB: 0.92120 | XGB: 0.92437 | CAT: 0.92588 | ENS: 0.92766


Retrain all three models on the complete training set. Average predictions and clip to [0, 1]. Save final submission CSV with 41,778 rows, no nulls, demand range [0.0006, 1.0000].

In [15]:
X_full, y_full, X_test = train[features], train['demand'], test[features]

lgb_f = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=-1, num_leaves=63, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
lgb_f.fit(X_full, y_full); p_lgb_t = lgb_f.predict(X_test)

xgb_f = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=7, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_f.fit(X_full, y_full); p_xgb_t = xgb_f.predict(X_test)

cat_f = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=7, random_seed=42, verbose=0)
cat_f.fit(X_full, y_full); p_cat_t = cat_f.predict(X_test)

preds = np.clip((p_lgb_t + p_xgb_t + p_cat_t) / 3.0, 0.0, 1.0)
submission = pd.DataFrame({'Index': test['Index'], 'demand': preds})
submission.to_csv(sub_path, index=False)
print(f'Saved. Shape: {submission.shape} | Nulls: {submission.isnull().sum().sum()} | Range: [{preds.min():.4f}, {preds.max():.4f}]')

Saved. Shape: (41778, 2) | Nulls: 0 | Range: [0.0006, 1.0000]
